In [2]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

import json
import pickle
import torch

from scipy.stats import wilcoxon

In [3]:
# Load data
df = pd.read_csv("All_DTE_results_semi_supervise.csv")

for col in df.columns:
    if col != 'Dataset':
        # Clean 
        df[col] = df[col].str.split("(").str[0].astype(float) / 100.0

In [4]:
np.mean(df['DTE-NP'])

0.8148526315789474

# This script calculate p-value of the best ensemble with FoMo-0D, DTE-NP, kNN, ICL, DTE-C



In [5]:
# strategy 1 - Biggest ensemble with 55 base models
strategy_1_ensemble_55_dir = "../results/adbench/strategy_1_Biggest_ensemble_55_models/train_seed=0"

# Best selected Bagging ensemble - strategy 2 (15 base models randomly)
strategy_2_ensemble_13_7_dir = "../results/adbench/strategy_2_ensemble_13_7/train_seed=0"

## 1. Best Baggig-ensemble

In [ ]:
# Strategy 1

ensemble_55_aucroc_scores = []
ensemble_55_aucpr_scores = []
ensemble_55_f1_scores = []
datasets_name = []
ensemble_55_time_per_instance = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(strategy_1_ensemble_55_dir):
    if "result.csv" in files:
        file_path = os.path.join(subdir, "result.csv")
        df = pd.read_csv(file_path)

        # extract dataset name
        dataset_name = os.path.basename(os.path.dirname(subdir))
        datasets_name.append(dataset_name)
        
        # extract scores
        aucroc = df['aucroc'].values[0]
        ensemble_55_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        ensemble_55_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        ensemble_55_f1_scores.append(f1)

        time_instance = df ['time-per-inst'].values[0]
        ensemble_55_time_per_instance.append(time_instance)

# compute average scores of all datasets
ensemble_55_avg_aucroc = sum(ensemble_55_aucroc_scores) / len(ensemble_55_aucroc_scores)
ensemble_55_avg_aucpr = sum(ensemble_55_aucpr_scores) / len(ensemble_55_aucpr_scores)
ensemble_55_avg_f1 = sum(ensemble_55_f1_scores) / len(ensemble_55_f1_scores)
ensemble_55_time_per_instance_avg = sum(ensemble_55_time_per_instance) / len(ensemble_55_time_per_instance)

print(f"number of datasets: {len(ensemble_55_aucroc_scores)}")
print(f"Average aucroc: {ensemble_55_avg_aucroc}")
print(f"Average aucpr: {ensemble_55_avg_aucpr}")
print(f"Average f1: {ensemble_55_avg_f1}")
print(f"Inference time per instance: {ensemble_55_time_per_instance_avg}")

number of datasets: 121
Average aucroc: 0.8149289254150415
Average aucpr: 0.5350642202580803
Average f1: 0.52948373909052
Inference time per instance: 4.773269885760852e-06


In [ ]:
bagging_ensemble_55_dict = {'dataset': datasets_name, 'aucroc': ensemble_55_aucroc_scores, 'aucpr': ensemble_55_aucpr_scores, 'f1': ensemble_55_f1_scores}
bagging_ensemble_55_df = pd.DataFrame(bagging_ensemble_55_dict)
#bagging_ensemble_55_df.to_csv('bagging_ensemble_55_result.csv')

## BEST Ensemble - Selected Bagging - ensemble_13_7  (containing 15 base models)

In [22]:
ensemble_13_7_aucroc_scores = []
ensemble_13_7_aucpr_scores = []
ensemble_13_7_f1_scores = []
datasets_name = []
ensemble_13_7_time_per_instance = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(strategy_2_ensemble_13_7_dir):
    if "result.csv" in files:
        file_path = os.path.join(subdir, "result.csv")
        df = pd.read_csv(file_path)

        # extract dataset name
        dataset_name = os.path.basename(os.path.dirname(subdir))
        datasets_name.append(dataset_name)
        
        # extract scores
        aucroc = df['aucroc'].values[0]
        ensemble_13_7_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        ensemble_13_7_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        ensemble_13_7_f1_scores.append(f1)

        time_instance = df ['time-per-inst'].values[0]
        ensemble_13_7_time_per_instance.append(time_instance)

# compute average scores of all datasets
ensemble_13_7_avg_aucroc = sum(ensemble_13_7_aucroc_scores) / len(ensemble_13_7_aucroc_scores)
ensemble_13_7_avg_aucpr = sum(ensemble_13_7_aucpr_scores) / len(ensemble_13_7_aucpr_scores)
ensemble_13_7_avg_f1 = sum(ensemble_13_7_f1_scores) / len(ensemble_13_7_f1_scores)
ensemble_13_7_time_per_instance_avg = sum(ensemble_13_7_time_per_instance) / len(ensemble_13_7_time_per_instance)

print(f"number of datasets: {len(ensemble_13_7_aucroc_scores)}")
print(f"Average aucroc: {ensemble_13_7_avg_aucroc}")
print(f"Average aucpr: {ensemble_13_7_avg_aucpr}")
print(f"Average f1: {ensemble_13_7_avg_f1}")
print(f"Inference time per instance: {ensemble_13_7_time_per_instance_avg}")

number of datasets: 121
Average aucroc: 0.8096822064386395
Average aucpr: 0.5227022034341368
Average f1: 0.5175232130889847
Inference time per instance: 4.74233431051334e-06


In [ ]:
selected_bagging_ensemble_13_7_dict = {'dataset': datasets_name, 'aucroc': ensemble_13_7_aucroc_scores, 'aucpr': ensemble_13_7_aucpr_scores, 'f1': ensemble_13_7_f1_scores}
selected_bagging_ensemble_13_7_df = pd.DataFrame(selected_bagging_ensemble_13_7_dict)
#selected_bagging_ensemble_13_7_df.to_csv('selected_bagging_ensemble_13_7_result.csv')

### 1. best_model with Fomo-0D

In [5]:
# new files:
bagging_ensemble_55_result = pd.read_csv('bagging_ensemble_55_result.csv')  #  #      processed_test_validation_ensemble_13_5.csv
default_fomo_result = pd.read_csv('default_fomo_result.csv')

bagging_ensemble_55_auc, default_fomo_auc = [], []
for ds, auc, default in zip(bagging_ensemble_55_result["dataset"],bagging_ensemble_55_result["aucroc"], default_fomo_result['aucroc']):
    bagging_ensemble_55_auc.append(auc)
    default_fomo_auc.append(default)

bagging_ensemble_55_auc = np.array(bagging_ensemble_55_auc)
default_fomo_auc = np.array(default_fomo_auc)

# Check differences
diffs = bagging_ensemble_55_auc - default_fomo_auc
print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# One-sided Wilcoxon test
stat, p_value = wilcoxon(bagging_ensemble_55_auc, default_fomo_auc, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)
print(f"mean value of bagging_ensemble_55 vs. fomo-0D: {np.mean(bagging_ensemble_55_auc)} vs. {np.mean(default_fomo_auc)}")  

Mean improvement: 0.055864374260867274
Better ratio: 0.8016528925619835
Statistic: 6153.0
p-value: 6.40706396831359e-13
mean value of bagging_ensemble_55 vs. fomo-0D: 0.8149289254150417 vs. 0.7590645511541743


The model shows a small average improvement in AUC (+0.017) and outperforms the baseline on 58% of datasets. However, the difference is not statistically significant (Wilcoxon signed-rank test, p = 0.127)

### 2. best_model with DTE-NP

In [ ]:
# new files:
bagging_ensemble_55_result = pd.read_csv('processed_bagging_ensemble_55_result.csv')  #  #      processed_test_validation_ensemble_13_5.csv
dte_np_df = df[['Dataset', 'DTE-NP']].copy()

def clean_name(x):
    x = str(x)
    x = re.sub(r"^\d+_", "", x)  # remove prefix like 10_
    x = x.lower()                # lowercase everything
    x = x.strip()                # remove spaces
    return x

bagging_ensemble_55_result["dataset_clean"] = bagging_ensemble_55_result["dataset"].apply(clean_name)
dte_np_df["dataset_clean"] = dte_np_df["Dataset"].apply(clean_name)
df_merged = pd.merge(bagging_ensemble_55_result, dte_np_df, on="dataset_clean")

# Extract paired values
auc1 = np.array(df_merged["aucroc"])
auc2 = np.array(df_merged["DTE-NP"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)
#print(f"mean value of mode_24_10 vs. DTE-NP: {np.mean(df_merged['aucroc'])} vs. {np.mean(df_merged['DTE-NP'])}")

Mean improvement: 0.01011304564369456
Better ratio: 0.5263157894736842
Statistic: 836.0
p-value: 0.4699166130287098
mean value of mode_24_10 vs. DTE-NP: 0.824965677222642 vs. 0.8148526315789473


### 3. best model kNN_semi

In [21]:
# new files:
bagging_ensemble_55_result = pd.read_csv('processed_bagging_ensemble_55_result.csv')  #  #      processed_test_validation_ensemble_13_5.csv
KNN_df = df[['Dataset', 'KNN']].copy()

bagging_ensemble_55_result["dataset_clean"] = bagging_ensemble_55_result["dataset"].apply(clean_name)
KNN_df["dataset_clean"] = KNN_df["Dataset"].apply(clean_name)
df_merged = pd.merge(bagging_ensemble_55_result, KNN_df, on="dataset_clean")

# Extract paired values
auc1 = np.array(df_merged["aucroc"])
auc2 = np.array(df_merged["KNN"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)


Mean improvement: 0.01640778248579982
Better ratio: 0.5964912280701754
Statistic: 932.0
p-value: 0.20095373170728725


### 4. best model with ICL

In [22]:
# new files:
bagging_ensemble_55_result = pd.read_csv('processed_bagging_ensemble_55_result.csv')  #  #      processed_test_validation_ensemble_13_5.csv
ICL_df = df[['Dataset', 'ICL']].copy()

bagging_ensemble_55_result["dataset_clean"] = bagging_ensemble_55_result["dataset"].apply(clean_name)
ICL_df["dataset_clean"] = ICL_df["Dataset"].apply(clean_name)
df_merged = pd.merge(bagging_ensemble_55_result, ICL_df, on="dataset_clean")

# Extract paired values
auc1 = np.array(df_merged["aucroc"])
auc2 = np.array(df_merged["ICL"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)


Mean improvement: 0.03142006318755419
Better ratio: 0.6140350877192983
Statistic: 1045.0
p-value: 0.021962812103084345


### 5. best model with DTE-C

In [23]:
# new files:
bagging_ensemble_55_result = pd.read_csv('processed_bagging_ensemble_55_result.csv')  #  #      processed_test_validation_ensemble_13_5.csv
DTE_C_df = df[['Dataset', 'DTE-C']].copy()

bagging_ensemble_55_result["dataset_clean"] = bagging_ensemble_55_result["dataset"].apply(clean_name)
DTE_C_df["dataset_clean"] = DTE_C_df["Dataset"].apply(clean_name)
df_merged = pd.merge(bagging_ensemble_55_result, DTE_C_df, on="dataset_clean")

# Extract paired values
auc1 = np.array(df_merged["aucroc"])
auc2 = np.array(df_merged["DTE-C"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)


Mean improvement: 0.02128146669632613
Better ratio: 0.6666666666666666
Statistic: 1141.0
p-value: 0.006231294463797087
